# Figure 2: Collocation accuracy on ADS-B flights

Note, the ADS-B data to create the figures is too large for GitHub and takes a while to download/process. Instead, I already ran this and saved the results to `../data/figure_2_adsb/figure_2_adsb_collocations.csv`. If you want to try it on different ADS-B data, feel free to uncomment that section  (you do need the x and y annotations of the aircraft, Sentinel-2 satellite `base_url`, `granule_id`, and `sensing_time`, and the ADS-B data).

In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import pyproj
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import FancyArrowPatch
from matplotlib import gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
import seaborn as sns

from pycontrails.datalib import sentinel
from pycontrails.datalib.leo_utils import correction
from iagos_toolkit.flight.aircraft_performance import create_flight_from_adsb

In [ ]:
def closest_point_on_polyline(px, py, xs, ys):
    """
    Given a point P(px,py) and a polyline (xs, ys),
    return the closest point (cx, cy) on the line.
    """
    xs = np.asarray(xs)
    ys = np.asarray(ys)

    best_dist = np.inf
    best_x0 = None
    best_y0 = None

    for i in range(len(xs)-1):
        x1, y1 = xs[i], ys[i]
        x2, y2 = xs[i+1], ys[i+1]

        # Segment vector
        ABx = x2 - x1
        ABy = y2 - y1

        # Vector AP
        APx = px - x1
        APy = py - y1

        # Project AP onto AB
        length_sq = ABx**2 + ABy**2
        if length_sq == 0:
            # Degenerate segment
            x0, y0 = x1, y1
        else:
            t = (APx*ABx + APy*ABy) / length_sq
            t = np.clip(t, 0, 1)
            x0 = x1 + t * ABx
            y0 = y1 + t * ABy

        dist = np.hypot(px - x0, py - y0)

        if dist < best_dist:
            best_dist = dist
            best_x0 = x0
            best_y0 = y0

    return best_x0, best_y0


def compute_ontrack_offtrack_signed(
    x_manual, y_manual,
    x_pred, y_pred,
    x_proj, y_proj,
    heading
):
    """
    Signed errors using heading:
    - On-track: positive if along heading, negative otherwise
    - Off-track: positive if right-hand side of heading
    """

    # Closest point on polyline (from predicted position)
    cx, cy = closest_point_on_polyline(x_pred, y_pred, x_proj, y_proj)

    # Heading unit vector
    h = np.array([np.cos(heading), np.sin(heading)])

    # --- On-track (manual → closest point) ---
    v_on = np.array([x_manual - cx, y_manual - cy])
    ontrack_mag = np.linalg.norm(v_on)
    ontrack_sign = np.sign(np.dot(v_on, h))
    ontrack = ontrack_sign * ontrack_mag

    # --- Off-track (predicted → closest point) ---
    v_off = np.array([x_pred - cx, y_pred - cy])
    offtrack_mag = np.linalg.norm(v_off)

    # 2D cross product for left/right
    cross = h[0] * v_off[1] - h[1] * v_off[0]
    offtrack_sign = np.sign(cross)
    offtrack = offtrack_sign * offtrack_mag

    return ontrack, offtrack, (cx, cy)


def create_circle(radius):
    t = np.linspace(0, 2*np.pi, 1000)
    x = np.cos(t) * radius
    y = np.sin(t) * radius

    return x, y

In [ ]:
# # Collect data for plotting

# df = pd.read_csv("../data/figure_2_adsb/adsb_metadata_summary.csv")

# plot_data = []

# for idx, row in tqdm(df.iterrows(), total=len(df)):
#     try:
#         x_true = row["x"]
#         y_true = row["y"]
        
#         handler = sentinel.Sentinel(
#             row["base_url"],
#             row["granule_id"],
#             bands=["B02", "B03", "B04"]
#         )


#         flight = create_flight_from_adsb(
#             f"../data/figure_2_adsb/{row['csv_path']}", "B788"
#         )

#         x_pred, y_pred, t_pred = handler.colocate_flight(flight)

#         df_flight = flight.dataframe  # or flight.data, depending on API
#         times = pd.to_datetime(df_flight["time"]).astype("int64") / 1e9
#         longitudes = df_flight["longitude"].values
#         latitudes = df_flight["latitude"].values

#         sensing_time = pd.to_datetime(row["sensing_time"]).value / 1e9

#         # Interpolate longitude and latitude
#         longitude = np.interp(sensing_time, times, longitudes)
#         latitude = np.interp(sensing_time, times, latitudes)

#         # add the x and y coordinates in the UTM coordinate system
#         utm_crs = handler.get_crs()

#         # Project from WGS84 to the x and y coordinates in the UTM coordinate system
#         transformer = pyproj.Transformer.from_crs("EPSG:4326", utm_crs, always_xy=True)
#         x_naive, y_naive = transformer.transform(longitude, latitude)

#         x1, y1 = transformer.transform(df_flight["longitude"], df_flight["latitude"])
#         df_flight.loc[:, "x"] = x1
#         df_flight.loc[:, "y"] = y1

#         ds_viewing_angles = handler.get_viewing_angle_metadata()
#         x_proj, y_proj = correction.scan_angle_correction(ds_viewing_angles, df_flight["x"], df_flight["y"], df_flight["altitude"], maxiter=3)
#         df_flight.loc[:, "x_proj"] = x_proj
#         df_flight.loc[:, "y_proj"] = y_proj

#         heading = row["heading"]
#         heading_rad = np.deg2rad(90.0 - heading)

#         ontrack, offtrack, (cx, cy) = compute_ontrack_offtrack_signed(
#             x_true, y_true,
#             x_pred, y_pred,
#             df_flight["x_proj"], df_flight["y_proj"],
#             heading_rad
#         )

#         ontrack_naive, offtrack_naive, (cx, cy) = compute_ontrack_offtrack_signed(
#             x_true, y_true,
#             x_naive, y_naive,
#             df_flight["x_proj"], df_flight["y_proj"],
#             heading_rad
#         )

#         # Store data
#         plot_data.append({
#             "x_true": x_true,
#             "y_true": y_true,
#             "x_pred": x_pred,
#             "y_pred": y_pred,
#             "x_naive": x_naive,
#             "y_naive": y_naive,
#             "ontrack_error": ontrack,
#             "offtrack_error": offtrack,
#             "ontrack_error_naive": ontrack_naive,
#             "offtrack_error_naive": offtrack_naive,
#             "heading": heading,
#             "base_url": row["base_url"]
#         })

#     except Exception as e:
#         print(e)

# df = pd.DataFrame(plot_data)
# df.to_csv("../data/figure_2_adsb/figure_2_adsb_collocations.csv")

In [ ]:
df = pd.read_csv("../data/figure_2_adsb/figure_2_adsb_collocations.csv")
df

In [ ]:
extent = [
    751900.0,
    771890.0,
    5776560.0,
    5796550.0
]

x = 764704.5484061396
y = 5786356.587301588

image_location = "../data/figure_2_adsb/example.png"

img = mpimg.imread(image_location)

# get the predicted aircraft location
handler = sentinel.Sentinel(
    "gs://gcp-public-data-sentinel-2/tiles/28/U/GC/S2B_MSIL1C_20220116T120359_N0301_R066_T28UGC_20220116T124838.SAFE",
    "L1C_T28UGC_A025403_20220116T120353",
    bands=["B02", "B03", "B04"]
)

flight = create_flight_from_adsb(
    f"../data/figure_2_adsb/48506d_L1C_T28UGC_A025403_20220116T120353.csv", "B788"
)

# Extract time series
df_flight = flight.dataframe  # or flight.data, depending on API
times = pd.to_datetime(df_flight["time"]).astype("int64") / 1e9
longitudes = df_flight["longitude"].values
latitudes = df_flight["latitude"].values

sensing_time = pd.to_datetime("2022-01-16 12:06:55.828000+00:00").value / 1e9

# Interpolate longitude and latitude
longitude = np.interp(sensing_time, times, longitudes)
latitude = np.interp(sensing_time, times, latitudes)

# add the x and y coordinates in the UTM coordinate system
utm_crs = handler.get_crs()

# Project from WGS84 to the x and y coordinates in the UTM coordinate system
transformer = pyproj.Transformer.from_crs("EPSG:4326", utm_crs, always_xy=True)
x_naive, y_naive = transformer.transform(longitude, latitude)
print(x_naive, y_naive)

x_pred, y_pred, sensing_time = handler.colocate_flight(flight)
print(x_pred, y_pred)

# add the x and y coordinates in the UTM coordinate system
x1, y1 = transformer.transform(df_flight["longitude"], df_flight["latitude"])
df_flight.loc[:, "x"] = x1
df_flight.loc[:, "y"] = y1

# project the x and y location to the image level
ds_viewing_angles = handler.get_viewing_angle_metadata()
x_proj, y_proj = correction.scan_angle_correction(ds_viewing_angles, df_flight["x"], df_flight["y"], df_flight["altitude"], maxiter=3)
df_flight.loc[:, "x_proj"] = x_proj
df_flight.loc[:, "y_proj"] = y_proj

In [ ]:
num_cols = ["x_true", "y_true", "x_pred", "y_pred", "x_naive", "y_naive"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

# Compute absolute errors
df["error_naive"] = np.sqrt((df["x_true"] - df["x_naive"])**2 + (df["y_true"] - df["y_naive"])**2)
df["error_pred"] = np.sqrt((df["x_true"] - df["x_pred"])**2 + (df["y_true"] - df["y_pred"])**2)

# 95th percentiles
p95_naive = df["error_naive"].quantile(0.95)
p95_pred = df["error_pred"].quantile(0.95)
print("Naive error 95th percentile:", p95_naive)
print("Predicted error 95th percentile:", p95_pred)

In [ ]:
colors = {
    "naive": "#E03C31",
    "pred": "#009B77",
    "circle_250": "magenta",
    "circle_1500": "darkturquoise",
    "arrow": "yellow",
    "flight_path": "white"
}

plt.rcParams.update({
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14
})

fig = plt.figure(figsize=(17, 10), dpi=300)

gs = gridspec.GridSpec(
    2, 2,
    width_ratios=[1, 2.6],
    height_ratios=[1, 1],
    wspace=-0.2,
    hspace=0.3
)

ax1 = fig.add_subplot(gs[0, 0])  # Histogram top-left
ax2 = fig.add_subplot(gs[1, 0])  # Scatter bottom-left
ax3 = fig.add_subplot(gs[:, 1])  # RGB image spans right

# --------------------------
# (a) Histogram
bin_size = 100
max_error = max(df["error_naive"].max(), df["error_pred"].max())
bins = np.arange(0, max_error + bin_size, bin_size)

ax1.hist(df["error_naive"], bins=bins, histtype="stepfilled", facecolor=colors["naive"], alpha=0.3,
         edgecolor="none", label="Naive interpolation")
ax1.hist(df["error_naive"], bins=bins, histtype="step", linewidth=1.5, color=colors["naive"])
ax1.hist(df["error_pred"], bins=bins, histtype="stepfilled", facecolor=colors["pred"], alpha=0.3,
         edgecolor="none", label="With corrections")
ax1.hist(df["error_pred"], bins=bins, histtype="step", linewidth=1.5, color=colors["pred"])

ax1.set_xticks([0, 1000, 2000, 3000, 4000])
ax1.set_xlabel("Absolute Error (m)")
ax1.set_ylabel("Count (-)")
ax1.set_xlim(0, df["error_naive"].max())
ax1.set_ylim(0, 125)
ax1.legend(fontsize=14)

ax1.axvline(250, color=colors["circle_250"], linestyle='dashed', linewidth=2)
ax1.axvline(1500, color=colors["circle_1500"], linestyle='dashed', linewidth=2)
ax1.set_box_aspect(1)
ax1.text(0.0, 1.07, "(a)", transform=ax1.transAxes, fontsize=14, fontweight='bold', va='top')
ax1.text(650, 90, "250m", ha="right", va="top", fontsize=14, color=colors["circle_250"], rotation=90)
ax1.text(1900, 94, "1500m", ha="right", va="top", fontsize=14, color=colors["circle_1500"], rotation=90)

# --------------------------
# (b) Scatter
ax2.scatter(df["ontrack_error_naive"] / 1000, df["offtrack_error_naive"] / 1000,
            alpha=0.6, color=colors["naive"], label="Naive interpolation", zorder=3)
ax2.scatter(df["ontrack_error"] / 1000, df["offtrack_error"] / 1000,
            alpha=0.6, color=colors["pred"], label="With corrections", zorder=4)

ax2.plot(*create_circle(1500), linestyle="--", linewidth=2, color=colors["circle_1500"])
ax2.plot(*create_circle(250), linestyle="--", linewidth=2, color=colors["circle_250"])

ax2.axhline(0, color='black', linewidth=1)
ax2.axvline(0, color='black', linewidth=1)
ax2.set_xlabel(r"$\Delta s$ (km)")
ax2.set_ylabel(r"$\Delta n$ (km)")
ax2.set_xlim(-4, 4)
ax2.set_ylim(-4, 4)
ax2.set_xticks([-4, -2, 0, 2, 4])
ax2.set_yticks([-4, -2, 0, 2, 4])
ax2.tick_params(labelsize=14)
ax2.grid(True, linestyle="--", alpha=0.6)
ax2.set_box_aspect(1)
ax2.text(0.0, 1.07, "(b)", transform=ax2.transAxes, fontsize=14, fontweight='bold', va='top')

# --------------------------
# (c) Example figure with annotations
ax3.imshow(img, extent=extent, origin="upper")
ax3.scatter(x_naive, y_naive, alpha=0.8, s=50, color=colors["naive"], label="Naive interpolation", zorder=4)
ax3.scatter(x_pred, y_pred, alpha=0.8, s=50, color=colors["pred"], label="With corrections", zorder=4)
ax3.plot(df_flight["x_proj"], df_flight["y_proj"], color=colors["flight_path"], linewidth=2, linestyle="--", alpha=0.9, zorder=1)

# Circles around point
ax3.plot(create_circle(1500)[0]+x, create_circle(1500)[1]+y, linestyle="--", linewidth=2, color=colors["circle_1500"], zorder=2)
ax3.plot(create_circle(250)[0]+x, create_circle(250)[1]+y, linestyle="--", linewidth=2, color=colors["circle_250"], zorder=2)

ax3.set_xlim(x - 4000, x + 4000)
ax3.set_ylim(y - 4000, y + 4000)

# Options for annotations arrows
heading_rad = 2.05 + np.pi      # radians
length_n = 1540                  # cross-track length (meters)
length_s = -2450                 # along-track length (meters)
length_n_start = 200              # location to start cross-track arrow (meters)
length_s_start = -200              # location to start along-track arrow (meters)

cross_direction = np.array([np.cos(heading_rad), np.sin(heading_rad)])
along_direction = np.array([np.cos(heading_rad - np.pi/2), np.sin(heading_rad - np.pi/2)])

# Cross-track
p1 = np.array([x_naive, y_naive]) + length_n_start * along_direction
p2 = p1 + length_n * cross_direction

dim_line = FancyArrowPatch(p1, p2, arrowstyle='|-|', linewidth=1, color=colors["arrow"], mutation_scale=5)
ax3.add_patch(dim_line)
ax3.text(x_naive - 50, y_naive - 800, r"$\Delta n$", ha="right", va="top", fontsize=14, color=colors["arrow"], zorder=2)

# Along track
p1_along = np.array([x_naive, y_naive]) + length_s_start * cross_direction
p2_along = p1_along + length_s * along_direction

dim_line = FancyArrowPatch(p1_along, p2_along, arrowstyle='|-|', linewidth=1, color=colors["arrow"], mutation_scale=5)
ax3.add_patch(dim_line)
ax3.text(x_naive + 1000, y_naive + 1200, r"$\Delta s$", ha="right", va="top", fontsize=14, color=colors["arrow"], zorder=2)

ax3.legend(loc="upper right", fontsize=14)
ax3.grid(True, linestyle="--", alpha=0.5)
ax3.axis('off')
ax3.text(0.0, 1.03, "(c)", transform=ax3.transAxes, fontsize=14, fontweight='bold', va='top')
ax3.annotate("Projected \nflight path",
             xy=(x - 2000, y - 1300),
             xytext=(x - 2000, y - 2300),
             arrowprops=dict(arrowstyle="->", linewidth=2, color="white", alpha=0.8),
             fontsize=14,
             alpha=0.8,
             color="white",
             ha="center",
             va="center")

# Zoom-in inset of aircraft
axins = inset_axes(ax3, width="30%", height="30%",
                   bbox_to_anchor=(-0.02, -0.68, 1., 1.),
                   bbox_transform=ax3.transAxes,
                   borderpad=0)

axins.scatter(x_pred, y_pred, alpha=0.8, s=80, color=colors["pred"], label="With corrections", zorder=4)
axins.imshow(img, extent=extent, origin="upper")
axins.plot(create_circle(250)[0]+x, create_circle(250)[1]+y, linestyle="--", linewidth=2, color=colors["circle_250"], zorder=1)
axins.set_xlim(x - 500, x + 500)
axins.set_ylim(y - 500, y + 500)
axins.set_aspect("equal")
axins.set_xticks([])
axins.set_yticks([])
for spine in axins.spines.values():
    spine.set_edgecolor("white")
    spine.set_linewidth(1.2)
mark_inset(ax3, axins, loc1=1, loc2=3, ec="dimgrey", linestyle="--", lw=1.5)

ax3.text(x + 2000, y - 800, "1500m", ha="right", va="top", fontsize=14, color=colors["circle_1500"])
axins.text(x + 400, y - 250, "250m", ha="right", va="top", fontsize=14, color=colors["circle_250"])

plt.savefig("../codecheck/figures/fig02.png", dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.show()